In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
BACKUP_DIR = '/content/drive/MyDrive/ai_detector_project'
os.makedirs(BACKUP_DIR, exist_ok=True)
print("Backup dir ready:", BACKUP_DIR)
print(os.listdir(BACKUP_DIR))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Backup dir ready: /content/drive/MyDrive/ai_detector_project
['raid_preprocessed.csv', 'raid_balanced_final.csv', 'model_hc3_plus_raid', 'hart_preprocessed.csv']


In [ ]:
import os

# حط التوكن الجديد هنا (من غير ما تشاركه مع حد أو تصوره)
os.environ["KAGGLE_API_TOKEN"] = "KGAT_4bd07f598a3919df8a621e238465b2d3"

!mkdir -p ~/.kaggle
!echo $KAGGLE_API_TOKEN > ~/.kaggle/access_token
!chmod 600 ~/.kaggle/access_token

!pip install -q kaggle
print("Kaggle API ready.")

Kaggle API ready.


In [ ]:
!kaggle datasets download -d mkhaleddeaf/hart-for-ai-detection -p /content/hart_data --unzip
!ls -la /content/hart_data

Dataset URL: https://www.kaggle.com/datasets/mkhaleddeaf/hart-for-ai-detection
License(s): CC0-1.0
100% 130M/130M [00:03<00:00, 37.0MB/s]

total 16
drwxr-xr-x 4 root root 4096 Sep  3 05:46 .
drwxr-xr-x 1 root root 4096 Sep  3 05:43 ..
drwxr-xr-x 2 root root 4096 Sep  3 05:41 hart_csv
drwxr-xr-x 6 root root 4096 Sep  3 05:41 hart_raw


In [ ]:
import pandas as pd
import glob

csv_files = glob.glob('/content/hart_data/hart_csv/*.csv')
print("CSV files found:", csv_files)

# هياخد أول ملف csv يلاقيه — لو فيه أكتر من ملف، غيّر الـ index هنا يدويًا
raw_df = pd.read_csv(csv_files[0])
print("\nShape:", raw_df.shape)
print("\nColumns:", list(raw_df.columns))
print("\nSample row:")
print(raw_df.head(2))

CSV files found: ['/content/hart_data/hart_csv/test.csv', '/content/hart_data/hart_csv/hart_essay.csv', '/content/hart_data/hart_csv/val.csv', '/content/hart_data/hart_csv/train.csv', '/content/hart_data/hart_csv/hart_dev_full.csv', '/content/hart_data/hart_csv/hart_writing.csv', '/content/hart_data/hart_csv/hart_arxiv.csv', '/content/hart_data/hart_csv/hart_news.csv']

Shape: (1200, 10)

Columns: ['id', 'domain', 'text', 'label', 'model', 'content_source', 'language_source', 'task_level1', 'task_level2', 'task_level3']

Sample row:
                           id domain  \
0  essay-AAATRP14318000682525  essay   
1      gen/arxiv-2107.06132v1  arxiv   

                                                text  label  \
0  In the article\r\n\r\n"Making Mona Lisa Smile"...  human   
1  The rapid advancement of Earth Observation (EO...     ai   

                        model                      content_source  \
0                       human                               human   
1  claude-3-

In [ ]:
# محاولة تلقائية لتحديد عمود النص وعمود المصدر
text_col_candidates = ['text', 'content', 'generation', 'essay', 'response', 'output', 'body']
source_col_candidates = ['content_source', 'model', 'source', 'label_source']

text_col = next((c for c in text_col_candidates if c in raw_df.columns), None)
source_col = next((c for c in source_col_candidates if c in raw_df.columns), None)

print("Detected text column:", text_col)
print("Detected source column:", source_col)

if text_col is None or source_col is None:
    print("\n⚠️ لم يتم التعرف تلقائيًا على الأعمدة الصحيحة.")
    print("افحص القائمة دي وحدد الاسم الصح يدويًا في الخلية اللي بعد كده:")
    print(list(raw_df.columns))

Detected text column: text
Detected source column: content_source


In [ ]:
# غيّر القيم دي لو الخلية اللي فاتت طلعت None أو أسماء غلط
# text_col = 'العمود اللي فيه النص'
# source_col = 'العمود اللي فيه human/machine:...'

print("Using text_col =", text_col)
print("Using source_col =", source_col)

Using text_col = text
Using source_col = content_source


In [ ]:
df = raw_df[[text_col, source_col]].copy()
df.columns = ['generation', 'source']

df['label'] = df['source'].apply(lambda x: 0 if str(x).strip().lower() == 'human' else 1)

df = df.dropna(subset=['generation'])
df = df[df['generation'].str.strip().str.len() > 20]
df = df.drop_duplicates(subset='generation')

print(df.shape)
print(df['label'].value_counts())
print("\nBreakdown by source:")
print(df['source'].value_counts())

# Backup فورًا
df.to_csv(BACKUP_DIR + '/hart_preprocessed.csv', index=False)
print("\nSaved backup:", BACKUP_DIR + '/hart_preprocessed.csv')

(1200, 3)
label
0    600
1    600
Name: count, dtype: int64

Breakdown by source:
source
human                                 600
machine:qwen2.5-72b-instruct          112
machine:claude-3-5-sonnet-20241022    106
machine:gemini-1.5-pro-002            106
machine:gpt-3.5-turbo-0125            106
machine:gpt-4o-2024-11-20              86
machine:llama-3.3-70b-instruct         84
Name: count, dtype: int64

Saved backup: /content/drive/MyDrive/ai_detector_project/hart_preprocessed.csv


In [ ]:
import os

# غيّر ده لو اسم الفولدر مختلف عندك
model_source_dir = BACKUP_DIR + '/model_hc3_plus_raid'

if not os.path.exists(model_source_dir):
    print("⚠️ مش لاقي الموديل في:", model_source_dir)
    print("الفولدرات الموجودة فعليًا في Backup dir:")
    print(os.listdir(BACKUP_DIR))
else:
    print("✅ Model found at:", model_source_dir)

✅ Model found at: /content/drive/MyDrive/ai_detector_project/model_hc3_plus_raid


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_path = model_source_dir
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print("Model loaded on:", device)

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Model loaded on: cuda


In [ ]:
from sklearn.model_selection import train_test_split
from datasets import Dataset

train_df, val_df = train_test_split(
    df, test_size=0.15, stratify=df['label'], random_state=42
)
train_ds = Dataset.from_pandas(train_df.reset_index(drop=True))
val_ds = Dataset.from_pandas(val_df.reset_index(drop=True))

def tokenize_fn(batch):
    return tokenizer(batch['generation'], truncation=True, padding='max_length', max_length=512)

train_ds = train_ds.map(tokenize_fn, batched=True)
val_ds = val_ds.map(tokenize_fn, batched=True)
train_ds = train_ds.rename_column("label", "labels")
val_ds = val_ds.rename_column("label", "labels")
train_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
val_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
print("Ready:", len(train_ds), "train /", len(val_ds), "val")

Map:   0%|          | 0/1020 [00:00<?, ? examples/s]

Map:   0%|          | 0/180 [00:00<?, ? examples/s]

Ready: 1020 train / 180 val


In [ ]:
!pip uninstall torchvision -y

Found existing installation: torchvision 0.26.0+cu128
Uninstalling torchvision-0.26.0+cu128:
  Successfully uninstalled torchvision-0.26.0+cu128


In [ ]:
import os
os.kill(os.getpid(), 9)

In [ ]:
from transformers import TrainingArguments, Trainer
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds),
        "precision": precision_score(labels, preds),
        "recall": recall_score(labels, preds),
    }

args = TrainingArguments(
    output_dir="./hart_continued_ft",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=2,
    learning_rate=5e-6,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=25,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=True,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.962745,0.497175,0.838889,0.857143,0.769912,0.966667
2,0.775644,0.621288,0.861111,0.874372,0.798165,0.966667
3,0.466180,0.715206,0.861111,0.874372,0.798165,0.966667
4,0.306170,0.796081,0.855556,0.870000,0.790909,0.966667


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
results = trainer.evaluate()
print(results)

Training Loss,Validation Loss,Epoch,Accuracy,F1,Precision,Recall
1.158697,0.532186,2,0.833333,0.852941,0.763158,0.966667


{'eval_loss': 0.5321856141090393, 'eval_accuracy': 0.8333333333333334, 'eval_f1': 0.8529411764705882, 'eval_precision': 0.7631578947368421, 'eval_recall': 0.9666666666666667}


In [ ]:
model.save_pretrained("./model_hc3_raid_hart")
tokenizer.save_pretrained("./model_hc3_raid_hart")

import shutil
final_model_backup = BACKUP_DIR + '/model_hc3_raid_hart'
if os.path.exists(final_model_backup):
    shutil.rmtree(final_model_backup)
shutil.copytree("./model_hc3_raid_hart", final_model_backup)
print("✅ Model backed up to:", final_model_backup)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Model backed up to: /content/drive/MyDrive/ai_detector_project/model_hc3_raid_hart
